# Destiny Studio — production autonome Colab Pro

Un seul notebook : Atlas → Blender → narration **Kokoro française ou anglaise** → sous-titres réellement alignés → Remotion → MP4 H.264/AAC dans Drive. **Aucun job GitHub Actions n’est requis.** Les modèles Oryx/Rhulk/Nezarec/Cropta restent des prototypes ; Athéon/Templier sont préservés.

Sur Android Chrome : ouvrez Colab, connectez-vous, choisissez **Exécution → Modifier le type d’exécution → GPU T4** si disponible. Exécutez les cellules dans l’ordre. Un GPU demandé n’est pas une validation GPU : le backend effectif figure dans les rapports. Le repli CPU est explicite et peut être lent.

Première installation : téléchargements réseau des outils et modèles vocaux. Aucun PC local, clé API ou terminal n’est nécessaire. Ne lancez pas une longue vidéo avant le smoke-test de 2 secondes.

In [ ]:
# @title 1. Monter Google Drive et installer le studio autonome
from google.colab import drive, output
from pathlib import Path
import hashlib, json, os, subprocess, sys, tarfile, urllib.request
from IPython.display import display, Video, FileLink
output.enable_custom_widget_manager()
drive.mount('/content/drive')
projet_a_reprendre = '' # @param {type:"string"}
prepared_pilot_only = False # @param {type:"boolean"}
DRIVE_ROOT = Path('/content/drive/MyDrive')
REPO = Path('/content/Destiny-Explained-Studio')
BRANCH = 'feat/atheon-performance-cinematic-direction'
def run(args, **kwargs):
    subprocess.run([str(x) for x in args], check=True, **kwargs)
if not REPO.exists():
    run(['git','clone','--depth','1','--single-branch','--branch',BRANCH,'https://github.com/Alexanst54/Destiny-Explained-Studio.git',REPO])
else:
    actual = subprocess.check_output(['git','-C',REPO,'branch','--show-current'],text=True).strip()
    assert actual in [BRANCH,''], 'Dossier sur une autre branche : utilisez un runtime neuf.'
if projet_a_reprendre:
    import re
    project=Path(projet_a_reprendre).expanduser().resolve()
    assert project.is_relative_to(DRIVE_ROOT.resolve()), 'Projet attendu dans Drive.'
    manifest=project/'project.json'
    if not manifest.exists(): manifest=project/'smoke/project.json'
    saved=json.loads(manifest.read_text());revision=saved.get('sourceCommit','')
    assert re.fullmatch(r'[0-9a-f]{40}',revision) and not saved.get('sourceDirty',True), 'Source de projet non épinglée : utilisez un nouveau projet.'
    run(['git','-C',REPO,'fetch','--depth','1','origin',revision])
    run(['git','-C',REPO,'checkout','--detach',revision])
print('Source épinglée dans ce runtime :',subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'],text=True).strip())
run(['apt-get','update','-qq'])
asound='libasound2t64' if subprocess.run(['apt-cache','show','libasound2t64'],capture_output=True).returncode==0 else 'libasound2'
run(['apt-get','install','-y','-qq','ffmpeg','espeak-ng','python3-venv','libxi6','libxrender1','libxkbcommon0','libsm6','libice6','libgl1','libegl1','libnss3','libgbm1','fonts-dejavu-core',asound])
def install_archive(url, checksum_url, name, destination):
    archive=Path('/content')/name
    urllib.request.urlretrieve(url,str(archive))
    checks=urllib.request.urlopen(checksum_url).read().decode()
    digest=hashlib.file_digest(archive.open('rb'),'sha256').hexdigest()
    assert any(digest in line and name in line for line in checks.splitlines()), 'Téléchargement invalide : SHA-256 incorrect.'
    # Official, SHA-verified archives only.
    run(['tar','-xf',archive,'-C',destination])
    archive.unlink()
NODE_DIR=Path('/content/node-v22.23.4-linux-x64')
if not NODE_DIR.exists():
    base='https://nodejs.org/dist/v22.23.4/'
    install_archive(base+'node-v22.23.4-linux-x64.tar.xz',base+'SHASUMS256.txt','node-v22.23.4-linux-x64.tar.xz','/content')
os.environ['PATH']=str(NODE_DIR/'bin')+':'+os.environ['PATH']
os.environ['HF_HOME']='/content/destiny-model-cache/huggingface'
os.environ['DESTINY_MODEL_CACHE']='/content/destiny-model-cache'
NODE=NODE_DIR/'bin/node'
BLENDER=Path('/content/blender-4.3.2-linux-x64/blender')
if not BLENDER.exists():
    base='https://download.blender.org/release/Blender4.3/'
    install_archive(base+'blender-4.3.2-linux-x64.tar.xz',base+'blender-4.3.2.sha256','blender-4.3.2-linux-x64.tar.xz','/content')
VENV=Path('/content/destiny-studio-venv')
if not VENV.exists(): run([sys.executable,'-m','venv',VENV])
PYTHON=VENV/'bin/python'
if not prepared_pilot_only:
    run([PYTHON,'-m','pip','install','--no-cache-dir','--index-url','https://download.pytorch.org/whl/cpu','torch==2.6.0','torchaudio==2.6.0'])
    run([PYTHON,'-m','pip','install','--no-cache-dir','-r',REPO/'remotion/studio/requirements.txt'])
else:
    print('Prepared Atheon pilot only: original six WAVs will be reused; Torch/Kokoro model installation is not needed. To use arbitrary Studio narration later, uncheck this option and rerun installation.')
run(['npm','ci'],cwd=REPO/'remotion')
run([NODE,REPO/'remotion/node_modules/@remotion/cli/remotion-cli.js','browser','ensure'],cwd=REPO/'remotion')
try: run(['nvidia-smi'])
except (FileNotFoundError,subprocess.CalledProcessError): print('ATTENTION : GPU NVIDIA indisponible. Blender utilisera le CPU si nécessaire.')
sys.path.insert(0,str(REPO/'remotion/studio'))
from colab_ui import form
print('Installation terminée. Aucun appel à GitHub Actions.')


In [ ]:
# @title 2. Choisir le boss et enregistrer le projet dans Drive
# Les listes d’arène, d’animation et de caméra changent avec le boss.
STUDIO = form(DRIVE_ROOT)


Cliquez **Enregistrer le projet** après vos choix. Le dossier contient `selection.json`, puis un manifeste immuable, la voix, les frames, les journaux et le MP4.

Pour reprendre un projet existant sans ressaisir le formulaire, utilisez la cellule suivante. Changer la narration, durée, qualité, sélection ou source exige un nouveau nom de projet ; on ne réutilise jamais silencieusement des frames incompatibles.

In [ ]:
# @title 2b. Reprendre un projet Drive existant (facultatif)
reprendre = False # @param {type:"boolean"}
dossier = '/content/drive/MyDrive/Destiny-Studio/destiny-001' # @param {type:"string"}
if reprendre:
    saved=Path(dossier).expanduser().resolve()
    assert saved.is_relative_to(DRIVE_ROOT.resolve()), 'Le projet doit être dans votre Drive.'
    assert (saved/'selection.json').exists(), 'selection.json introuvable.'
    STUDIO={'folder':str(saved),'config':str(saved/'selection.json')}
    print('Projet restauré :',saved)


In [ ]:
# @title 3. Smoke-test obligatoire — 2 secondes, narration multilingue et MP4
assert STUDIO.get('config'), 'Enregistrez ou restaurez un projet avant cette étape.'
COMMAND=[PYTHON,REPO/'remotion/studio/production.py',STUDIO['config'],STUDIO['folder'],'--blender',BLENDER,'--node',NODE]
try:
    run(COMMAND+['--smoke'])
    SMOKE=Path(STUDIO['folder'])/'smoke'
    display(Video(str(SMOKE/'video.mp4'),embed=True))
    print(json.loads((SMOKE/'validation.json').read_text()))
except Exception as error:
    print('Smoke-test interrompu. Corrigez la cause puis relancez cette même cellule. Les caches valides sont conservés. Journaux :',Path(STUDIO['folder'])/'smoke/logs')
    raise


**Inspectez le smoke-test avant de poursuivre** : bonne identité, cadrage, mouvement, parole audible dans la langue choisie et sous-titres synchronisés. Le smoke utilise la phrase courte « Voici ce boss. », le même boss/animation/caméra, Draft et résolution réduite au même ratio (320×240 pour le réglage Preview). Il ne valide pas la qualité finale ou le matériel T4 à lui seul.

La narration complète sera synthétisée et alignée avant les images de production. Si elle dépasse la durée, aucune image de production n’est rendue : choisissez un nouveau projet avec une durée adaptée. Les effets et matériaux ne perdent pas de géométrie en basse résolution : LOD0 reste explicite.

In [ ]:
# @title 4. Production complète ou reprise — après inspection du smoke-test
smoke_inspecte = False # @param {type:"boolean"}
assert smoke_inspecte, 'Visionnez le smoke-test et cochez smoke_inspecte.'
assert STUDIO.get('config'), 'Enregistrez ou restaurez un projet.'
try:
    run([PYTHON,REPO/'remotion/studio/production.py',STUDIO['config'],STUDIO['folder'],'--blender',BLENDER,'--node',NODE])
except Exception:
    print('Production interrompue. Consultez les logs dans Drive puis relancez cette cellule sans modifier le projet. Les PNG complets sont vérifiés avant réemploi ; une frame corrompue est recalculée.')
    raise


In [ ]:
# @title 5. Vérifier, visionner et télécharger le MP4 final
from google.colab import files
FOLDER=Path(STUDIO['folder'])
report=json.loads((FOLDER/'validation.json').read_text())
assert report.get('status')=='passed', 'La validation finale ne passe pas : consultez les logs.'
# Revalidate the actual file before displaying or downloading it.
from production import validate, verify, sha
config=validate(json.loads((FOLDER/'selection.json').read_text()))
verify(FOLDER/'video.mp4',config)
assert sha(FOLDER/'video.mp4')==report['sha256'], 'MP4 modifié ou incomplet.'
print(json.dumps(report,ensure_ascii=False,indent=2))
display(Video(str(FOLDER/'video.mp4'),embed=True))
telecharger = False # @param {type:"boolean"}
if telecharger: files.download(str(FOLDER/'video.mp4'))
print('Copie permanente déjà enregistrée dans Drive :',FOLDER/'video.mp4')


In [ ]:
# @title 6. Acceptation matérielle T4 (facultative, après un court rendu réel)
executer_acceptation_t4 = False # @param {type:"boolean"}
exiger_reprise_chaude = False # @param {type:"boolean"}
lecture_android_verifiee = False # @param {type:"boolean"}
if executer_acceptation_t4:
    FOLDER=Path(STUDIO['folder'])
    reports=[json.loads(p.read_text()) for p in sorted((FOLDER/'blender').glob('chunk-*/progress.json'))]
    config=validate(json.loads((FOLDER/'selection.json').read_text()))
    frames=[f for r in reports for f in r['frames'].values()]
    assert len(frames)==config['frames'], 'Séquence incomplète.'
    assert all(f['device']['selected']=='GPU' and f['device']['backend'] in ['CUDA','OPTIX'] for f in frames), 'Certaines frames sont CPU ou utilisent un autre backend : T4 non validée.'
    assert all(any('T4' in name for name in f['device']['devices']) for f in frames), 'Appareil réellement utilisé différent d’une T4.'
    verified=verify(FOLDER/'video.mp4',config)
    hits=sum(r['cacheHits'] for r in reports)
    if exiger_reprise_chaude:
        assert hits==config['frames'] and sum(r['renderedThisRun'] for r in reports)==0, 'Relancez la cellule 4 à paramètres identiques, puis recommencez.'
    acceptance={'status':'passed','actualT4Rendering':True,'video':verified,'warmResumeVerified':bool(exiger_reprise_chaude),'androidPlaybackReviewedByUser':bool(lecture_android_verifiee),'sourceCommit':json.loads((FOLDER/'project.json').read_text())['sourceCommit']}
    (FOLDER/'t4-acceptance.json').write_text(json.dumps(acceptance,ensure_ascii=False,indent=2))
    print(acceptance)
else:
    print('Aucune validation T4 exécutée. Le succès CPU local ne vaut pas test T4/Android.')


In [ ]:
# @title Pilot Episode 01 — English audio-only preparation (no full video render)
prepare_english_pilot = False # @param {type:"boolean"}
pilot_drive_folder = '/content/drive/MyDrive/Destiny-Studio/atheon-explained-en' # @param {type:"string"}
if prepare_english_pilot:
    pilot_folder=Path(pilot_drive_folder).expanduser().resolve()
    assert pilot_folder.is_relative_to(DRIVE_ROOT.resolve()), 'Choose a folder inside mounted Drive.'
    run([PYTHON,REPO/'remotion/studio/pilot.py',REPO/'episodes/atheon-explained-en/script.json',pilot_folder])
    timing=json.loads((pilot_folder/'timing.json').read_text())
    print('English title:',timing['title'])
    print('Measured duration:',timing['durationSeconds'],'seconds; 85–95s target met:',timing['targetMet'])
    print('Six English scene selections and synchronized word timings saved to Drive. No Blender frames or combined episode were rendered.')
else:
    print('Full Atheon episode deferred to the user’s later Colab Tesla T4 production session.')


## Récupération et acceptation Colab / T4

- Session interrompue : renseignez le chemin du projet dans `projet_a_reprendre` de la cellule 1 ; elle restaure le commit enregistré. Reconnectez Drive, puis utilisez la restauration 2b et relancez la même étape. Gardez le même commit, projet et Blender. Aucun cache ne dépend de GitHub Actions. Une modification de source est refusée : épinglez le commit de `project.json` ou utilisez un nouveau dossier.
- GPU : vérifiez les noms des appareils et `backend` OPTIX/CUDA dans `validation.json` et les logs. Le rapport distingue chaque frame réellement rendue sur CPU/GPU, y compris après repli. L’affichage `nvidia-smi` seul n’est pas une preuve de rendu GPU.
- Reprise : relancez la production terminée et vérifiez les `cacheHits` de chaque `blender/chunk-*/progress.json`. Les PNG valides ne sont pas recalculés. Après corruption volontaire d’une **copie de test**, une seule frame doit être recalculée.
- Sur T4 réelle : smoke-test, inspection audio/visuelle Android, clip court Balanced, interruption/reprise, vérification du MP4 et du backend effectif. Archivez les rapports et votre résultat d’inspection dans Drive. Un test CPU local n’est pas une validation Colab/T4/Android.
- Musique : facultative, uniquement un fichier que vous fournissez avec sa provenance/licence ; aucune musique du jeu n’est téléchargée.
- Pas assez d’espace / erreur réseau / mémoire GPU : corrigez la cause, choisissez un nouveau projet plus léger si les réglages changent. Une erreur de modèle ou d’alignement ne déclenche aucun faux modèle ni sous-titre inventé.

Guide complet et limites : `docs/production-studio/README.md` dans le dépôt. Les anciens épisodes, modèles et narrations restent inchangés.

# Render Complete Atheon Episode

**Tonight's short route:** run installation cell 1 with `prepared_pilot_only = True` (skip model/voice downloads), then use only the five controls below. Do not run the old audio-preparation cell: this path reuses the reviewed six-scene package, original English WAVs, word timestamps and SRT already committed.

Default: **Balanced, 360×640, vertical 9:16, 30 FPS, LOD0**, 2672 real Blender frames, **89.067 seconds**. Higher resolution can cost much more time. The original narration is never cut, regenerated or accelerated. No GitHub Actions are used.

Choose a T4 GPU runtime if available. Preflight probes the effective Cycles device and checks the pilot assets, clean source commit, tools and reported Drive/local space. The Drive mount may not expose remote account quota: check your available Google Drive storage too. No finishing-time guarantee is made before real GPU frames are measured.

The **smoke-test is the complete first scene (11.5 seconds)**, not an extra throwaway render. Its 345 frames are reused when rendering all scenes. Watch it and listen before continuing. Every completed scene can resume independently after a disconnection.


In [ ]:
# @title A. Prepare complete episode — fast preflight and restore
complete_episode_folder = '/content/drive/MyDrive/Destiny-Studio/atheon-episode-01-360-balanced' # @param {type:"string"}
complete_resolution = 'Fast 360×640' # @param ['Fast 360×640','540×960','720×1280','1080×1920']
complete_quality = 'balanced' # @param ['balanced','draft','cinematic']
restore_saved_settings = True # @param {type:"boolean"}
allow_cpu_rendering = False # @param {type:"boolean"}
import signal
assert all(name in globals() for name in ['REPO','DRIVE_ROOT','PYTHON','BLENDER','NODE']), 'Run installation cell 1 first.'
EPISODE=Path(complete_episode_folder).expanduser().resolve()
assert EPISODE.is_relative_to(DRIVE_ROOT.resolve()) and EPISODE!=DRIVE_ROOT.resolve(), 'Choose a dedicated project subfolder inside Drive.'
assert (REPO/'remotion/studio/complete_atheon.py').exists(), 'This checkout predates the complete-episode section. For a new project, delete/reconnect the runtime and rerun installation. For resume, restore the saved project commit in cell 1.'
from complete_atheon import RESOLUTIONS
cw,ch=RESOLUTIONS[complete_resolution]
project_file=EPISODE/'project.json'
if restore_saved_settings and project_file.exists():
    saved=json.loads(project_file.read_text())
    assert saved.get('identity',{}).get('kind')=='complete-atheon-v1', 'This is not a complete-episode project; choose a new folder.'
    opts=saved['identity']['options'];cw,ch=opts['width'],opts['height'];complete_quality=opts['quality']
    print('Restored saved settings:',cw,ch,complete_quality,'— choose a NEW folder to change them; existing cache is preserved.')
EPISODE_ARGS=[str(PYTHON),str(REPO/'remotion/studio/complete_atheon.py'),str(EPISODE),'--width',str(cw),'--height',str(ch),'--quality',complete_quality,'--blender',str(BLENDER),'--node',str(NODE)]
if allow_cpu_rendering: EPISODE_ARGS+=['--allow-cpu']
def episode_action(action):
    args=EPISODE_ARGS[:2]+[action]+EPISODE_ARGS[2:]
    proc=subprocess.Popen(args,start_new_session=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in proc.stdout: print(line,end='',flush=True)
        result=proc.wait()
    except KeyboardInterrupt:
        try: os.killpg(proc.pid,signal.SIGINT)
        except ProcessLookupError: pass
        try: proc.wait(timeout=40)
        except subprocess.TimeoutExpired:
            try: os.killpg(proc.pid,signal.SIGKILL)
            except ProcessLookupError: pass
            proc.wait()
        print('Stopped. Keep this Drive folder and resume the same action; validated frames are retained.')
        raise
    finally:
        proc.stdout.close()
    if result: raise RuntimeError('Episode action failed. Read reports/activity.json and logs/ in Drive, correct the cause, then rerun this action without changing settings.')
episode_action('prepare')
print('Prepared:',EPISODE,'\nNo image frames have been rendered by this step.')
print('On reconnection: set projet_a_reprendre in installation cell 1 to this folder, then rerun A with the same folder. Saved settings are restored automatically.')


In [ ]:
# @title B. Smoke-test — first complete scene, reused in the episode
assert 'EPISODE_ARGS' in globals(), 'Run A. Prepare first.'
episode_action('smoke')
smoke_report=json.loads((EPISODE/'atheon-smoke.validation.json').read_text())
assert smoke_report.get('status')=='passed'
display(Video(str(EPISODE/'atheon-smoke.mp4'),embed=True,width=270))
print('Original full first-scene narration, English titles/subtitles. Actual frame devices:',smoke_report['deviceReport'])
print('If framing, voice or render time is unsuitable, stop and choose a NEW folder/settings. Otherwise continue; these 345 frames will not be rendered again.')


In [ ]:
# @title C. Render all six scenes — or resume interrupted scenes
first_scene_reviewed = False # @param {type:"boolean"}
assert first_scene_reviewed, 'Watch and listen to the first-scene smoke-test, then confirm review.'
assert 'EPISODE_ARGS' in globals(), 'Run A. Prepare first.'
episode_action('render')
print('All scene frame caches complete in Drive. Next: assemble the continuous MP4. Re-running this step validates and reuses completed frames.')


In [ ]:
# @title D. Assemble and validate the complete 89-second episode with Remotion
assert 'EPISODE_ARGS' in globals(), 'Run A. Prepare first.'
episode_action('assemble')
final_report=json.loads((EPISODE/'atheon-episode-01.validation.json').read_text())
assert final_report.get('status')=='passed' and final_report.get('fullEpisodeRendered') is True
print({k:final_report[k] for k in ['duration','frames','fps','width','height','videoCodec','audioCodec','subtitleWords','deviceReport']})
print('Saved:',EPISODE/'atheon-episode-01.mp4','\nReport:',EPISODE/'atheon-episode-01.validation.json')
print('Assembly uses the original six WAVs and every real frame. No frame interpolation, narration acceleration or scene trimming.')


In [ ]:
# @title E. Validate again, preview and download
assert 'EPISODE_ARGS' in globals(), 'Run A. Prepare first.'
episode_action('validate')
from google.colab import files
final_video=EPISODE/'atheon-episode-01.mp4'
if final_video.stat().st_size<=20*1024**2:
    display(Video(str(final_video),embed=True,width=270))
else:
    print('Large video: open it in Drive or download; inline embedding skipped for mobile responsiveness.')
download_complete_mp4 = False # @param {type:"boolean"}
download_validation_report = False # @param {type:"boolean"}
if download_complete_mp4: files.download(str(final_video))
if download_validation_report: files.download(str(EPISODE/'atheon-episode-01.validation.json'))
print('Permanent Drive copy:',final_video)


## Complete episode recovery

- Runtime disconnected: reconnect Drive in cell 1. Set `projet_a_reprendre` to the complete-episode folder so the saved clean source commit is restored; use `prepared_pilot_only = True`. Rerun A with the same folder, then B/C/D as needed. B and C reuse validated frames; D reuses an already validated final export.
- Keep `project.json`, `public/scenes/`, `scenes/`, reports and logs. Do not delete caches between scenes. Interrupted chunks resume from valid PNGs; completed scenes are independently checked by request/source fingerprint and PNG decode/SHA. A corrupt PNG is rendered again; other valid PNGs are retained.
- Do not change resolution/quality in an active project. Use a new folder if changing settings. Previous content is not deleted. Default 360×640 Balanced prioritizes completion; ETA is rough and comes from measured frames, not a promised T4 speedup.
- Missing GPU: select a GPU runtime and retry. CPU rendering requires explicitly enabling it and can be very slow. **Assembly/validation of completed frames can use CPU without enabling CPU scene rendering.**
- The final report checks H.264/AAC, dimensions, 30 FPS, 2672 decoded frames, duration, original English WAV hashes, subtitle bounds and the placement/presence of audio in all six scenes. Human review of pronunciation, image quality and final Android playback is still needed.
- No real Colab/T4 acceptance is claimed by this notebook's development tests. Actual rendered backend/device counts appear in the saved reports. The general standalone Studio cells above remain available; uncheck `prepared_pilot_only` in installation if you want to synthesize new narration instead of using the prepared pilot.


# T4 performance lab — benchmark before rerendering

The successful user run took about 20 hours at 540×960 / 2672 frames, OptiX, with approximately 10% observed utilization. That observation does not identify the bottleneck. **Do not start a complete rerender yet.** This opt-in lab compares the exact current renderer, instrumented Drive writes, `/content` scratch, persistent scene reuse, samples/denoising and a PNG/thread experiment on the **same 30–60 representative frames per case**. Cases are sequential on the same runtime. The existing Studio and complete-episode cells above are preserved.

Run installation cell 1 with `prepared_pilot_only=True` on a fresh GPU T4 runtime. Use a new dedicated benchmark folder; never point this at the successful episode. Results include real paired PNGs, a mobile HTML comparison gallery, GPU samples, raw logs, cache/Drive/build/render/save timing and a labeled benchmark montage. This is not a replacement episode. Sample/denoiser changes require visual review. No six-hour promise or speedup is assumed.


In [ ]:
# @title P0. Diagnose the successful original episode without rerendering (optional)
inspect_completed_episode = False # @param {type:"boolean"}
completed_episode_folder = '' # @param {type:"string"}
existing_diagnostic_report = '/content/drive/MyDrive/Destiny-Studio/atheon-existing-diagnostic.json' # @param {type:"string"}
verify_all_cached_pngs = False # @param {type:"boolean"}
if inspect_completed_episode:
    assert completed_episode_folder.startswith('/content/drive/MyDrive/'), 'Select your completed original Drive episode.'
    args=[PYTHON,REPO/'remotion/studio/inspect_existing.py',completed_episode_folder,existing_diagnostic_report]
    if verify_all_cached_pngs: args+=['--verify-cache']
    run(args)
    print('Read-only diagnosis saved outside the original project:',existing_diagnostic_report)


In [ ]:
# @title P1. Plan the paired T4 benchmark (no render yet)
benchmark_folder = '/content/drive/MyDrive/Destiny-Studio/atheon-t4-benchmark-01' # @param {type:"string"}
benchmark_frames = 30 # @param [30,36,48,60] {type:"raw"}
benchmark_resolution = '540×960' # @param ['360×640','540×960']
assert 'REPO' in globals() and 'BLENDER' in globals(), 'Run installation cell 1 first.'
assert benchmark_folder.startswith('/content/drive/MyDrive/'), 'Use a new Drive benchmark folder.'
import signal, subprocess, json
from pathlib import Path
bw,bh=map(int,benchmark_resolution.split('×'))
BENCHMARK_ARGS=[str(PYTHON),str(REPO/'remotion/studio/performance.py'),benchmark_folder,'--scratch','/content/atheon-benchmark','--frames',str(benchmark_frames),'--width',str(bw),'--height',str(bh),'--blender',str(BLENDER),'--node',str(NODE)]
def lab_run(args):
    process=subprocess.Popen([str(x) for x in args],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,start_new_session=True)
    try:
        for line in process.stdout: print(line.rstrip(),flush=True)
        if process.wait(): raise RuntimeError('Command failed. Read the preserved logs; resume with identical settings/source.')
    except BaseException:
        try: os.killpg(process.pid,signal.SIGINT)
        except ProcessLookupError: pass
        try: process.wait(timeout=40)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid,signal.SIGKILL);process.wait()
        raise
lab_run(BENCHMARK_ARGS)
print('Plan saved. Next cell is an explicit benchmark, not a full episode render.')


In [ ]:
# @title P2. Execute the paired benchmark on the actual T4
run_t4_benchmark = False # @param {type:"boolean"}
assert run_t4_benchmark, 'Opt in after checking the plan and free Drive space.'
assert 'BENCHMARK_ARGS' in globals(), 'Run P1 first.'
lab_run(BENCHMARK_ARGS+['--execute'])
summary=json.loads((Path(benchmark_folder)/'benchmark.json').read_text())
for row in summary['estimates']: print(row)
print('Inspect comparison.html, visual-differences.json, each result.json/export-result.json and GPU/log traces in:',benchmark_folder)
print('Cold T4 render-only projections exclude the full episode export and workload variance. Warm/resumed trials are excluded from speed ratios. Use a new folder for an independent cold repeat.')


# Cinematic Atheon — opt-in directed edition

This separate project preserves the six original WAVs, all words, 89.067-second timing and approved Atheon/Templar geometry. It adds 19 story-driven shots: arena establishment, monumental reveal, mechanisms, Oracle field, the existing Templar in its own arena, contrasting portal landscapes, anonymous Guardians, an Aegis-inspired light shield, and an empty-throne conclusion. Portal landscapes and silhouettes are editorial reconstruction, not an assertion of universe-wide Vex control.

**Review the benchmark and real shot previews before enabling a complete rerender.** The default candidate keeps Balanced 48 samples / existing denoising / LOD0, uses local scratch with 10-frame Drive checkpoints and persistent compatible scenes. This is a candidate configuration, not a measured speedup. At most nine new frames between successful checkpoints may need recomputing after abrupt VM loss; previous verified frames survive. Keep one runtime per project.


In [ ]:
# @title Q1. Prepare or resume a separate cinematic project
cinematic_folder = '/content/drive/MyDrive/Destiny-Studio/atheon-cinematic-01' # @param {type:"string"}
cinematic_resolution = '540×960' # @param ['360×640','540×960','720×1280']
assert 'lab_run' in globals(), 'Run P1 to initialize the streaming controls.'
assert cinematic_folder.startswith('/content/drive/MyDrive/'), 'Use a NEW dedicated Drive folder; preserve your successful export.'
cw,ch=map(int,cinematic_resolution.split('×'))
CINEMATIC_ARGS=[str(PYTHON),str(REPO/'remotion/studio/cinematic_atheon.py')]
CINEMATIC_OPTIONS=[cinematic_folder,'--scratch','/content/atheon-cinematic','--width',str(cw),'--height',str(ch),'--blender',str(BLENDER),'--node',str(NODE)]
def cinematic_action(action,extra=()): lab_run(CINEMATIC_ARGS+[action]+CINEMATIC_OPTIONS+list(extra))
cinematic_action('prepare')


In [ ]:
# @title Q2. Render real shot previews (stills, not motion acceptance)
cinematic_action('preview')
print('Review every genuine Blender still in Drive:',cinematic_folder+'/previews/index.html')
from IPython.display import Image, display
for preview in sorted((Path(cinematic_folder)/'previews/scenes').glob('*/frames/shot-*/frame-0000.png')):
    display(Image(filename=str(preview),width=216))


In [ ]:
# @title Q3. Render directed scenes — only after benchmark and art review
paired_benchmark_reviewed = False # @param {type:"boolean"}
all_shot_previews_reviewed = False # @param {type:"boolean"}
scene_to_render = 'all' # @param ['all','01-hook','02-identity','03-vault','04-time','05-guardians','06-mystery']
assert paired_benchmark_reviewed and all_shot_previews_reviewed, 'Measure first, then inspect all shot previews before a complete rerender.'
extra=['--benchmark-reviewed','--reviewed-previews']
if scene_to_render!='all': extra+=['--scene',scene_to_render]
cinematic_action('render',extra)


In [ ]:
# @title Q4. Assemble, validate and download the directed edition
export_directed_episode = False # @param {type:"boolean"}
assert export_directed_episode, 'Enable only when all directed scene frames are complete.'
cinematic_action('assemble')
cinematic_action('validate')
from google.colab import files
files.download(cinematic_folder+'/atheon-episode-01-cinematic.mp4')
files.download(cinematic_folder+'/atheon-episode-01-cinematic.validation.json')


## Lab recovery and honest acceptance

A new runtime can restore a directed project's saved commit using `projet_a_reprendre` in installation cell 1; rerun P1 (initialization only), Q1 with the same saved resolution and Drive path, then the interrupted action. For a benchmark, put its Drive folder in `projet_a_reprendre` in installation cell 1 to restore its saved source commit; keep the exact source/settings or use a new run folder. Valid PNG/SHA checkpoints are restored from Drive into a new `/content` cache. Source mismatches are refused rather than overwriting earlier output. A warm benchmark is for recovery validation and is not a cold speed measurement.

No T4 speedup, six-hour completion, full cinematic export, or physical Android acceptance has been measured during development. Read `docs/production-studio/PERFORMANCE-CINEMATIC.md` for the benchmark protocol, known measurement boundaries, visual review checklist and PR #17 analysis. Do not run GitHub Actions.
